# Voicebox Backend on Google Colab T4

Este notebook levanta **todo el backend** en Colab para que Qwen3-TTS genere audio en remoto.

## Flujo
1. Configurar parámetros
2. Instalar dependencias
3. (Opcional) montar Google Drive
4. Levantar backend en segundo plano
5. Exponer URL pública con ngrok
6. Validar con smoke tests


In [ ]:
!nvidia-smi

In [ ]:
# ===== Configuracion =====
REPO_URL = "https://github.com/cdryampi/voicebox.git"
REPO_REF = "main"  # branch/tag/commit

# Seguridad: NO pongas claves reales en este archivo.
# Puedes dejar estos valores en None y se pediran en runtime.
VOICEBOX_API_KEY = None
VOICEBOX_GROQ_API_KEY = None  # opcional
NGROK_AUTH_TOKEN = None       # opcional

# Persistencia
USE_GOOGLE_DRIVE = False
DRIVE_DATA_DIR = "/content/drive/MyDrive/voicebox-data"
LOCAL_DATA_DIR = "/content/voicebox-data"

# Backend
VOICEBOX_HOST = "0.0.0.0"
VOICEBOX_PORT = "17493"
VOICEBOX_DEFAULT_MODEL_SIZE = "0.6B"
VOICEBOX_ALLOWED_ORIGINS = "*"


In [ ]:
# ===== Clone + install =====
!git clone "$REPO_URL" /content/voicebox
%cd /content/voicebox
!git fetch --all --tags
!git checkout "$REPO_REF"

!python -m pip install --upgrade pip
!pip install -r backend/requirements.txt
!pip install pyngrok requests

In [ ]:
# ===== Entorno y storage =====
import os
from pathlib import Path
from getpass import getpass


def _read_colab_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value and str(value).strip():
            return str(value).strip()
    except Exception:
        pass
    return None


def _resolve_secret(name: str, explicit_value, required: bool = False):
    if explicit_value is not None and str(explicit_value).strip():
        return str(explicit_value).strip()

    from_colab_secret = _read_colab_secret(name)
    if from_colab_secret:
        return from_colab_secret

    if required:
        entered = getpass(f"{name}: ").strip()
        if not entered:
            raise RuntimeError(f"{name} es obligatorio para despliegue remoto seguro")
        return entered

    return None


if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = DRIVE_DATA_DIR
else:
    data_dir = LOCAL_DATA_DIR

Path(data_dir).mkdir(parents=True, exist_ok=True)

api_key = _resolve_secret('VOICEBOX_API_KEY', VOICEBOX_API_KEY, required=True)
groq_api_key = _resolve_secret('VOICEBOX_GROQ_API_KEY', VOICEBOX_GROQ_API_KEY, required=False)
ngrok_auth_token = _resolve_secret('NGROK_AUTH_TOKEN', NGROK_AUTH_TOKEN, required=False)

os.environ['VOICEBOX_COLAB_PROFILE'] = 'true'
os.environ['VOICEBOX_HOST'] = VOICEBOX_HOST
os.environ['VOICEBOX_PORT'] = VOICEBOX_PORT
os.environ['VOICEBOX_DEFAULT_MODEL_SIZE'] = VOICEBOX_DEFAULT_MODEL_SIZE
os.environ['VOICEBOX_ALLOWED_ORIGINS'] = VOICEBOX_ALLOWED_ORIGINS
os.environ['VOICEBOX_DATA_DIR'] = data_dir
os.environ['VOICEBOX_API_KEY'] = api_key

if groq_api_key:
    os.environ['VOICEBOX_GROQ_API_KEY'] = groq_api_key

print('VOICEBOX_DATA_DIR =', os.environ['VOICEBOX_DATA_DIR'])
print('VOICEBOX_COLAB_PROFILE =', os.environ['VOICEBOX_COLAB_PROFILE'])
print('VOICEBOX_PORT =', os.environ['VOICEBOX_PORT'])
print('VOICEBOX_API_KEY configurada =', bool(os.environ.get('VOICEBOX_API_KEY')))
print('VOICEBOX_GROQ_API_KEY configurada =', bool(os.environ.get('VOICEBOX_GROQ_API_KEY')))


In [ ]:
import os
# ===== Start backend =====
import subprocess
import time
import requests

server = subprocess.Popen([
    'python', '-m', 'uvicorn',
    'backend.main:app',
    '--host', os.environ.get('VOICEBOX_HOST', '0.0.0.0'),
    '--port', os.environ.get('VOICEBOX_PORT', '17493')
])
print('Server PID:', server.pid)

base = f"http://127.0.0.1:{os.environ.get('VOICEBOX_PORT', '17493')}"
headers = {'Authorization': f"Bearer {os.environ['VOICEBOX_API_KEY']}"}

for i in range(60):
    try:
        r = requests.get(f"{base}/health", headers=headers, timeout=5)
        if r.status_code == 200:
            print('Backend ready:', base)
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Backend did not become ready in time')

In [ ]:
import os
# ===== Public URL via ngrok =====
from pyngrok import ngrok

if ngrok_auth_token:
    ngrok.set_auth_token(ngrok_auth_token)

port = int(os.environ.get('VOICEBOX_PORT', '17493'))
public_url = ngrok.connect(port, bind_tls=True).public_url

print('\n=== CONNECTION INFO ===')
print('Public URL:', public_url)
print('Server URL for app:', public_url)
print('API key: [HIDDEN]')
print('Header: Authorization: Bearer <API_KEY>')


In [ ]:
import os
# ===== Smoke tests =====
import requests

base = f"http://127.0.0.1:{os.environ.get('VOICEBOX_PORT', '17493')}"
headers = {'Authorization': f"Bearer {os.environ['VOICEBOX_API_KEY']}"}

for path in ['/health', '/runtime', '/models/status']:
    resp = requests.get(base + path, headers=headers, timeout=60)
    print(path, '->', resp.status_code)
    print(resp.json())
    print('---')


In [ ]:
import os
# ===== Optional: pre-cargar modelo 0.6B =====
# Ejecuta esta celda si quieres dejar el modelo listo antes de la primera generación.

import requests

base = f"http://127.0.0.1:{os.environ.get('VOICEBOX_PORT', '17493')}"
headers = {'Authorization': f"Bearer {os.environ['VOICEBOX_API_KEY']}"}

r = requests.post(f"{base}/models/load", params={'model_size': '0.6B'}, headers=headers, timeout=3600)
print('models/load ->', r.status_code)
print(r.json())


In [ ]:
# ===== Optional: shutdown backend =====
import signal

try:
    server.send_signal(signal.SIGTERM)
    print('Backend stopped')
except Exception as e:
    print('Shutdown error:', e)
